Імпорт пакетів

In [12]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Preprocessing
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Model
from sklearn.linear_model import LogisticRegression

# Metrics
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Всі пакети успішно імпортовані")

Всі пакети успішно імпортовані


Завантаження датасету

In [6]:
# Завантаження датасету Rain in Australia
# Датасет: https://www.kaggle.com/jsphyg/weather-dataset-rattle-package
# Файл: weatherAUS.csv

import pandas as pd
df = pd.read_csv('weatherAUS.csv')
print(f"✓ Датасет: {df.shape}")


✓ Датасет: (145460, 23)


3.1 Видалення ознак з великою кількістю пропущених значень

In [7]:
# Визначення порогу для видалення (більше 40% пропусків)
missing_threshold = 0.4
missing_percentage = df.isnull().sum() / len(df)

print("Відсоток пропущених значень по ознаках:")
print(missing_percentage.sort_values(ascending=False))

# Видалення ознак з великою кількістю пропусків
columns_to_drop = missing_percentage[missing_percentage > missing_threshold].index.tolist()
print(f"\nОзнаки для видалення: {columns_to_drop}")

df = df.drop(columns=columns_to_drop)
print(f"\nДатасет після видалення ознак: {df.shape}")

Відсоток пропущених значень по ознаках:
Sunshine         0.480098
Evaporation      0.431665
Cloud3pm         0.408071
Cloud9am         0.384216
Pressure9am      0.103568
Pressure3pm      0.103314
WindDir9am       0.072639
WindGustDir      0.070989
WindGustSpeed    0.070555
Humidity3pm      0.030984
WindDir3pm       0.029066
Temp3pm          0.024811
RainTomorrow     0.022460
Rainfall         0.022419
RainToday        0.022419
WindSpeed3pm     0.021050
Humidity9am      0.018246
WindSpeed9am     0.012148
Temp9am          0.012148
MinTemp          0.010209
MaxTemp          0.008669
Date             0.000000
Location         0.000000
dtype: float64

Ознаки для видалення: ['Evaporation', 'Sunshine', 'Cloud3pm']

Датасет після видалення ознак: (145460, 20)


In [8]:
# Видалення колонок Date та RainTomorrow з розгляду на цьому етапі
df_features = df.drop(columns=['Date', 'RainTomorrow'])

# Розділення на числові та категоріальні ознаки
numeric_features = df_features.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = df_features.select_dtypes(include=['object']).columns.tolist()

print("Числові ознаки:")
print(numeric_features)
print(f"\nКатегоріальні ознаки:")
print(categorical_features)

Числові ознаки:
['MinTemp', 'MaxTemp', 'Rainfall', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Temp9am', 'Temp3pm']

Категоріальні ознаки:
['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainToday']


In [9]:
# Конвертація Date в datetime
df['Date'] = pd.to_datetime(df['Date'])

# Створення ознак Year та Month
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

print("Приклад обробки часових ознак:")
print(df[['Date', 'Year', 'Month']].head())
print(f"\nРоки у датасеті: {sorted(df['Year'].unique())}")
print(f"Максимальний рік: {df['Year'].max()}")

Приклад обробки часових ознак:
        Date  Year  Month
0 2008-12-01  2008     12
1 2008-12-02  2008     12
2 2008-12-03  2008     12
3 2008-12-04  2008     12
4 2008-12-05  2008     12

Роки у датасеті: [np.int32(2007), np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017)]
Максимальний рік: 2017


In [10]:
# Year обробляється як числова ознака
numeric_features.append('Year')

# Month обробляється як категоріальна ознака
categorical_features.append('Month')

print(f"Оновлені числові ознаки: {numeric_features}")
print(f"Оновлені категоріальні ознаки: {categorical_features}")

Оновлені числові ознаки: ['MinTemp', 'MaxTemp', 'Rainfall', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Temp9am', 'Temp3pm', 'Year']
Оновлені категоріальні ознаки: ['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainToday', 'Month']


In [11]:
# Визначення максимального року
max_year = df['Year'].max()
min_year = df['Year'].min()

print(f"Діапазон років у датасеті: {min_year} - {max_year}")

# Булева індексація: тест = останній рік, тренування = решта років
train_mask = df['Year'] != max_year
test_mask = df['Year'] == max_year

# Розділення всього датасету
df_train = df[train_mask].copy()
df_test = df[test_mask].copy()

print(f"\nТренувальна вибірка: {df_train.shape[0]} рядків (роки: {sorted(df_train['Year'].unique())})")
print(f"Тестова вибірка: {df_test.shape[0]} рядків (рік: {sorted(df_test['Year'].unique())})")

# Видалення пропусків у цільовій змінній (RainTomorrow)
print(f"\nПропущені значення RainTomorrow до видалення:")
print(f"Тренування: {df_train['RainTomorrow'].isnull().sum()}")
print(f"Тест: {df_test['RainTomorrow'].isnull().sum()}")

df_train = df_train[df_train['RainTomorrow'].notna()]
df_test = df_test[df_test['RainTomorrow'].notna()]

print(f"\nЗагальний обсяг після видалення пропусків у RainTomorrow:")
print(f"Тренування: {df_train.shape[0]}")
print(f"Тест: {df_test.shape[0]}")

Діапазон років у датасеті: 2007 - 2017

Тренувальна вибірка: 136837 рядків (роки: [np.int32(2007), np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016)])
Тестова вибірка: 8623 рядків (рік: [np.int32(2017)])

Пропущені значення RainTomorrow до видалення:
Тренування: 3110
Тест: 157

Загальний обсяг після видалення пропусків у RainTomorrow:
Тренування: 133727
Тест: 8466
